In [1]:

import json
import pennylane as qml
import pennylane.numpy as np

def Sphi(phi):
    return np.array([[np.exp(1j * phi), 0], [0, np.exp(-1j * phi)]])

def U(a):
    return np.array([[a, np.sqrt(1 - a**2)], [np.sqrt(1 - a**2), -a]])

def matrix(angles, a):
    phi0, phi1, phi2, phi3 = angles
    return Sphi(phi0) @ U(a) @ Sphi(phi1) @ U(a) @ Sphi(phi2) @ U(a) @ Sphi(phi3)

def coefficients(angles):
    a_values = np.array([1, -1, 0.5, -0.5])  
    P_values = np.array([matrix(angles, a)[0, 0] for a in a_values])
    
    A = np.array([[1, 1], [1/8, 1/2]])
    b = np.array([P_values[0], P_values[2]])
    solution = np.linalg.solve(A, b)
    alpha, beta = solution
    
    return np.array([alpha, beta])

def run(test_case_input: str) -> str:
    ins = np.array(json.loads(test_case_input))
    coeffs = coefficients(ins)
    outs = [[np.real(elem).item(), np.imag(elem).item()] for elem in coeffs]
    return json.dumps(outs)

def check(solution_output: str, expected_output: str) -> None:
    solution_output = json.loads(solution_output)
    expected_output = json.loads(expected_output)
    assert np.allclose(solution_output, expected_output, rtol = 1e-4)

test_cases = [
    ('[0.5,0.8,1.0,1.0]', '[[-0.1707976, -2.4084889], [-0.8166822, 2.2507432]]'),
    ('[-0.20409113, -0.91173829, 0.91173829, 0.20409113]', '[[2.5,0],[-1.5,0]]')
]

for i, (input_, expected_output) in enumerate(test_cases):
    print(f"Running test case {i} with input '{input_}'...")

    try:
        output = run(input_)
    except Exception as exc:
        print(f"Runtime Error. {exc}")
    else:
        try:
            check(output, expected_output)
            print("Correct!")
        except AssertionError:
            print(f"Wrong Answer. Have: '{output}'. Want: '{expected_output}'.")


Running test case 0 with input '[0.5,0.8,1.0,1.0]'...
Correct!
Running test case 1 with input '[-0.20409113, -0.91173829, 0.91173829, 0.20409113]'...
Correct!
